# RAG Lab: Поиск по базе научных статей OpenAlex

В этой лабораторной работе мы будем работать с небольшой базой данных научных статей из репозитория [OpenAlex](https://openalex.org/), представленной в виде **Markdown-таблицы**. Мы сравним два подхода к ответам на вопросы по этой базе:

1. **RAG (Retrieval-Augmented Generation)** — семантический поиск по фрагментам таблицы с помощью векторного хранилища
2. **Tool Calling** — структурированный поиск по in-memory базе данных через вызов функций

## Архитектура

```
┌─────────────────────────────────────────────────────────────────┐
│                        Подход 1: RAG                           │
│  Markdown ──► Chunking ──► Vector Store ──► File Search Agent  │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                    Подход 2: Tool Calling                      │
│  Markdown ──► In-Memory DB ──► Pydantic Tools ──► Tool Agent   │
└─────────────────────────────────────────────────────────────────┘
```

## План работы

| Часть | Описание |
|-------|----------|
| 1 | Загрузка и исследование данных |
| 2 | Ручное чанкование и создание векторного хранилища |
| 3 | Проверка семантического поиска |
| 4 | Создание RAG-агента |
| 5 | Нетривиальные примеры с RAG |
| 6 | Tool Calling с Pydantic (in-memory база данных) |
| 7 | Сравнительная оценка: RAG vs Tool Calling |

## Настройка окружения

Установим необходимые библиотеки:

In [ ]:
%pip install --quiet openai python-dotenv tqdm

> **ВНИМАНИЕ**: После установки библиотек рекомендуется перезапустить Kernel ноутбука.

## Авторизация

Для работы с Yandex Cloud нам понадобится `folder_id` (идентификатор каталога) и `api_key` (API-ключ сервисного аккаунта). Мы предполагаем, что эти значения хранятся в файле `.env`, который можно скачать, выполнив ячейку ниже:

In [ ]:
!curl -o .env {{url_of_dotenv_file}}

Также скачаем реализацию класса `Agent` из репозитория в текущую директорию:

In [ ]:
!curl -o Agent.py https://raw.githubusercontent.com/yandex-ai-studio/ai-studio-course/refs/heads/main/4-rag-search/Agent.py

Загрузим переменные окружения и создадим клиент для работы с API. Мы импортируем класс `Agent` и функцию `create_client` из файла [Agent.py](../Agent.py):

In [ ]:
import os, sys, io, json, re
from glob import glob
from tqdm import tqdm
from IPython.display import Markdown, display
from dotenv import load_dotenv
from Agent import Agent, create_client

load_dotenv()

folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]
model = f"gpt://{folder_id}/qwen3-235b-a22b-fp8/latest"

client = create_client()

def printx(string):
    display(Markdown(string))

print(f"✅ Авторизация настроена (folder_id: {folder_id[:8]}...)")

---

## Часть 1: Загрузка и исследование данных

Наша база данных — это Markdown-файл `openalex_rag_snapshot.md`, содержащий таблицу научных статей из OpenAlex. Каждая строка таблицы описывает одну статью и содержит следующие поля:

| Поле | Описание |
|------|----------|
| OpenAlex ID | Уникальный идентификатор статьи |
| Title | Название статьи |
| Year | Год публикации |
| Venue | Место публикации (журнал, конференция) |
| Type | Тип публикации (article, preprint, review и т.д.) |
| Cited by | Количество цитирований |
| Open Access | Открытый доступ (yes/no) |
| Authors | Авторы |
| Top concepts | Ключевые концепции |
| Landing page | Ссылка на статью |
| Abstract | Реконструированная аннотация |

Скачаем файл в текущую директорию:

In [ ]:
!curl -o openalex_rag_snapshot.md https://raw.githubusercontent.com/yandex-ai-studio/ai-studio-course/main/data/openalex_rag_snapshot.md


**Задание:** Прочитайте файл и изучите его структуру. Выведите количество строк, заголовок таблицы и несколько первых записей. Посмотрите, какие столбцы есть в таблице.

In [ ]:
# Читаем файл с данными
data_path = "openalex_rag_snapshot.md"


---

## Часть 2: Ручное чанкование и создание векторного хранилища

Для работы RAG-агента необходимо загрузить данные в **векторное хранилище** (Vector Store). Однако наш файл — это одна большая Markdown-таблица, которую нельзя просто так разбить на части: каждый фрагмент должен содержать **заголовок таблицы**, чтобы LLM понимала, что означают данные в каждой колонке.

### Стратегия чанкования

1. Сохраняем **заголовок** таблицы (первые 2 строки)
2. Последовательно добавляем строки данных
3. Когда размер чанка превышает порог (~600 токенов ≈ 3000 символов), формируем файл и загружаем его
4. Каждый новый чанк начинается с заголовка

**Задание:** Создайте векторное хранилище:

In [ ]:
# Создаём векторное хранилище


Теперь выполним ручное чанкование таблицы. Каждый чанк будет содержать заголовок таблицы и несколько строк данных. Размер чанка примерно 600 токенов (один токен ≈ 5 символов).

> **ВАЖНО:** Не забудьте отключить автоматическую стратегию чанкования, установив значение `max_chunk_size_tokens` заведомо больше размера ручного чанка. В противном случае будут использоваться значения по умолчанию, и это может привести к двойному чанкованию!

Обратите внимание, что мы загружаем файлы в виде `BytesIO` — это позволяет не создавать временные файлы на диске:

In [ ]:
# Разбейте таблицу на фрагменты и загрузите их в вектороное хранилище
# Можно записать фрагменты на диск в виде файлов, или загружать их "на лету"
# используя client.files.create(file=(f"openalex_chunk_{i}.txt", io.BytesIO(s.encode("utf-8")), "text/markdown")
# Отключите чанкование при добавлении в векторное хранилище, указав
# заведомо больший размер чанка (2600), в противном случае будет применено
# чанкование по умолчанию

---

## Часть 3: Проверка семантического поиска

Прежде чем создавать агента, давайте убедимся, что семантический поиск по векторному хранилищу работает корректно. Мы выполним несколько поисковых запросов и посмотрим на результаты.

**Задание:** Выполните поиск по нескольким запросам и изучите возвращаемые результаты — файлы, оценки релевантности и содержимое.

In [ ]:
# Тест: Поиск статей по медицинской сегментации изображений


Убедитесь, что семантический поиск находит релевантные фрагменты таблицы по смыслу запроса. Обратите внимание, что каждый фрагмент содержит заголовок таблицы — это помогает модели понять структуру данных.

---

## Часть 4: Создание RAG-агента

Теперь создадим агента, который будет использовать файловый поиск по нашему векторному хранилищу для ответа на вопросы о научных статьях.

Для этого мы:
1. Опишем инструмент `file_search` в виде словаря
2. Создадим экземпляр `Agent` из модуля `Agent.py`
3. Зададим системный промпт, объясняющий агенту контекст данных


**Задание:** Создайте RAG-агента и проверьте, как он отвечает на различные вопросы.

In [ ]:
# Инструмент файлового поиска
file_search_tool = {
}

# Системный промпт для RAG-агента
rag_instruction = """
...
"""

rag_agent = Agent(
...
)

print("📖 RAG-агент готов!")

Проверим работу агента на простых вопросах:

In [ ]:
# Вопрос 1: Поиск статей по теме
result = rag_agent("Какие статьи в базе посвящены генеративно-состязательным сетям (GANs)?")
printx(result.output_text)

In [ ]:
# Вопрос 2: Информация о конкретной работе
result = rag_agent("Расскажи о статье Transformer-XL. Кто её авторы и сколько у неё цитирований?")
printx(result.output_text)

---

## Часть 5: Нетривиальные примеры с RAG

RAG хорошо работает для поиска конкретных статей по описанию. Однако давайте посмотрим, как он справляется с более сложными аналитическими запросами, требующими агрегации данных.

**Задание:** Попробуйте задать агенту вопросы, требующие подсчёта, сравнения или агрегации данных.

In [ ]:
# Нетривиальный вопрос 1: Подсчёт цитирований автора
result = rag_agent("Сколько суммарных цитирований у статей, в которых автором является Graham Neubig?")
printx(result.output_text)

In [ ]:
# Нетривиальный вопрос 2: Максимальное цитирование
result = rag_agent("Какая статья в базе имеет наибольшее количество цитирований? Укажи название, авторов и число цитирований.")
printx(result.output_text)

In [ ]:
# Нетривиальный вопрос 3: Фильтрация по критерию
result = rag_agent("Сколько статей типа 'review' есть в базе? Перечисли их названия.")
printx(result.output_text)

### Наблюдения

Обратите внимание на ограничения RAG-подхода:

- **Неполнота**: RAG извлекает только часть фрагментов из базы, поэтому подсчёт может быть неточным
- **Зависимость от качества поиска**: агент видит только те строки таблицы, которые вернул семантический поиск
- **Агрегация**: задачи, требующие полного перебора данных ("сколько всего...", "самая цитируемая..."), выполняются приблизительно

Именно для решения этих проблем мы реализуем альтернативный подход — **Tool Calling** с прямым доступом к данным.

---

## Часть 6: Tool Calling с Pydantic (In-Memory база данных)

В этом подходе мы:
1. Разберём Markdown-таблицу в Python-структуру (список словарей)
2. Определим **Pydantic-инструменты** для поиска и получения информации
3. Создадим агента, который использует эти инструменты для ответа на вопросы

Ключевое отличие от RAG: агент через инструменты получает **точный** доступ к данным, а не приблизительный через семантический поиск.

### Шаг 1: Парсинг Markdown-таблицы в структуру данных

**Задание:** Разберите Markdown-таблицу в список словарей, где каждый словарь — это одна статья.

In [ ]:
# Преобразуйте таблицу в список словарей `papers_db`

### Шаг 2: Определение Pydantic-инструментов

Создадим два инструмента:

1. **`SearchPapers`** — универсальный поиск по базе данных с необязательными фильтрами: ключевые слова, автор, год, тип публикации. Если указано несколько фильтров — они комбинируются (AND).
2. **`GetPaperCitations`** — получение информации о цитированиях для конкретной статьи по названию (или части названия).

Такая архитектура позволяет агенту самостоятельно комбинировать вызовы инструментов для решения сложных запросов. Например, чтобы узнать суммарные цитирования автора, агент может:
1. Вызвать `SearchPapers(author="Graham Neubig")` — получить все статьи автора
2. Суммировать цитирования из результатов

**Задание:** Реализуйте Pydantic-классы для инструментов.

In [ ]:
# Реализуйте и проверьте работу инструментов

### Шаг 3: Создание Tool Calling агента

Теперь создадим агента, который использует наши Pydantic-инструменты. Класс `Agent` из `Agent.py` автоматически конвертирует Pydantic-классы в JSON-схемы для Function Calling, а также обрабатывает вызовы функций в цикле.

**Задание:** Создайте агента с инструментами `SearchPapers` и `GetPaperCitations` и протестируйте его.

In [ ]:
# Системный промпт для Tool Calling агента
tc_instruction = """
Опишите в промпте доступные инструменты...
"""

tc_agent = Agent(
    ...
)


Проверим работу Tool Calling агента на тех же вопросах, которые мы задавали RAG-агенту:

In [ ]:
# Вопрос 1: Поиск статей по теме
result = tc_agent("Какие статьи в базе посвящены генеративно-состязательным сетям (GANs)?")
printx(result.output_text)

In [ ]:
# Вопрос 2: Подсчёт цитирований автора
tc_agent.reset()  # Сброс истории
result = tc_agent("Сколько суммарных цитирований у статей, в которых автором является Graham Neubig?")
printx(result.output_text)

In [ ]:
# Вопрос 3: Статьи типа review
tc_agent.reset()
result = tc_agent("Сколько статей типа 'review' есть в базе? Перечисли их названия.")
printx(result.output_text)

---

## Часть 7: Сравнительная оценка: RAG vs Tool Calling

Для объективного сравнения двух подходов создадим набор тестовых запросов с **заранее известными правильными ответами**, вычисленными программно по нашей базе данных.

### Шаг 1: Формирование тестового набора

**Задание:** Вычислите правильные ответы на основе разобранной таблицы, а затем проверьте, как каждый агент справляется с этими вопросами.

In [ ]:
# === Вычисляем ground truth ===

# Тест 1: Количество статей типа "review"
reviews = [p for p in papers_db if p.get("Type", "").lower() == "review"]
gt_review_count = len(reviews)
gt_review_titles = [p["Title"] for p in reviews]
print(f"✅ Тест 1: Статей типа 'review' = {gt_review_count}")
#for t in gt_review_titles:
#    print(f"   - {t}")

print()

# Тест 2: Самая цитируемая статья
top_paper = max(papers_db, key=lambda p: p.get("Cited by", 0))
gt_top_title = top_paper["Title"]
gt_top_citations = top_paper["Cited by"]
print(f"✅ Тест 2: Самая цитируемая = '{gt_top_title}' ({gt_top_citations} цитирований)")

print()

# Тест 3: Число статей Nils Reimers
reimers_papers = [p for p in papers_db if "Nils Reimers" in p.get("Authors", "")]
gt_reimers_count = len(reimers_papers)
gt_reimers_citations = sum(p.get("Cited by", 0) for p in reimers_papers)
print(f"✅ Тест 3: Статей Nils Reimers = {gt_reimers_count}, суммарные цитирования = {gt_reimers_citations}")

print()

# Тест 4: Количество статей 2024 года
papers_2024 = [p for p in papers_db if p.get("Year") == 2024]
gt_2024_count = len(papers_2024)
print(f"✅ Тест 4: Статей за 2024 год = {gt_2024_count}")

print()

# Тест 5: Количество статей с open access
oa_papers = [p for p in papers_db if p.get("Open Access", "").lower() == "yes"]
gt_oa_count = len(oa_papers)
print(f"✅ Тест 5: Статей с Open Access = {gt_oa_count}")

### Шаг 2: Определяем тестовый набор

Оформим тестовые вопросы и ожидаемые ответы в структуру данных:

In [ ]:
test_suite = [
    {
        "question": "Сколько статей типа 'review' есть в базе?",
        "expected_answer": str(gt_review_count),
        "description": "Подсчёт статей определённого типа"
    },
    {
        "question": "Какая статья имеет наибольшее количество цитирований? Напиши только название и число цитирований.",
        "expected_answer": f"{gt_top_title}, {gt_top_citations}",
        "description": "Поиск максимума"
    },
    {
        "question": f"Сколько статей написал Nils Reimers и сколько у них суммарных цитирований?",
        "expected_answer": f"{gt_reimers_count} статей, {gt_reimers_citations} цитирований",
        "description": "Фильтрация по автору + агрегация"
    },
    {
        "question": "Сколько статей в базе опубликовано в 2024 году?",
        "expected_answer": str(gt_2024_count),
        "description": "Фильтрация по году"
    },
    {
        "question": "Сколько статей имеют открытый доступ (Open Access = yes)?",
        "expected_answer": str(gt_oa_count),
        "description": "Подсчёт по полю Open Access"
    }
]

print(f"📋 Определено {len(test_suite)} тестов")
for i, t in enumerate(test_suite, 1):
    print(f"   {i}. {t['description']}: {t['expected_answer']}")

### Шаг 3: Запуск тестов

Запустим каждый тестовый вопрос через оба агента и сохраним результаты. Для удобства создадим вспомогательную функцию, которая проверяет, содержит ли ответ агента ожидаемое значение:

In [ ]:
def check_answer(agent_answer: str, expected: str) -> bool:
    """Проверяет, содержит ли ответ агента все ключевые элементы ожидаемого ответа."""
    # Извлекаем все числа из ожидаемого ответа
    expected_numbers = re.findall(r'\d+', expected)
    # Проверяем, что все числа присутствуют в ответе
    for num in expected_numbers:
        if num not in agent_answer:
            return False
    return True

results = []

for i, test in enumerate(test_suite, 1):
    print(f"\n{'='*60}")
    print(f"📝 Тест {i}: {test['description']}")
    print(f"❓ Вопрос: {test['question']}")
    print(f"✅ Ожидаемый ответ: {test['expected_answer']}")
    
    # RAG-агент
    rag_agent.reset()
    try:
        rag_result = rag_agent(test["question"])
        rag_answer = rag_result.output_text
    except Exception as e:
        rag_answer = f"Ошибка: {e}"
    rag_correct = check_answer(rag_answer, test["expected_answer"])
    
    print(f"\n🔍 RAG: {'✅' if rag_correct else '❌'}")
    print(f"   {rag_answer[:200]}..." if len(rag_answer) > 200 else f"   {rag_answer}")
    
    # Tool Calling агент
    tc_agent.reset()
    try:
        tc_result = tc_agent(test["question"])
        tc_answer = tc_result.output_text
    except Exception as e:
        tc_answer = f"Ошибка: {e}"
    tc_correct = check_answer(tc_answer, test["expected_answer"])
    
    print(f"\n🔧 Tool Calling: {'✅' if tc_correct else '❌'}")
    print(f"   {tc_answer[:200]}..." if len(tc_answer) > 200 else f"   {tc_answer}")
    
    results.append({
        "test": test["description"],
        "rag_correct": rag_correct,
        "tc_correct": tc_correct,
        "rag_answer": rag_answer[:100],
        "tc_answer": tc_answer[:100]
    })

### Шаг 4: Сводная таблица результатов

In [ ]:
# Формируем сводную таблицу
table = "| # | Тест | RAG | Tool Calling |\n"
table += "|---|------|-----|-------------|\n"

rag_score = 0
tc_score = 0

for i, r in enumerate(results, 1):
    rag_mark = "✅" if r["rag_correct"] else "❌"
    tc_mark = "✅" if r["tc_correct"] else "❌"
    rag_score += int(r["rag_correct"])
    tc_score += int(r["tc_correct"])
    table += f"| {i} | {r['test']} | {rag_mark} | {tc_mark} |\n"

table += f"| | **Итого** | **{rag_score}/{len(results)}** | **{tc_score}/{len(results)}** |\n"

printx(table)

### Выводы

Результаты наглядно демонстрируют различия между двумя подходами:

| Критерий | RAG (File Search) | Tool Calling |
|----------|-------------------|--------------|
| **Точность подсчёта** | Приблизительная (видит только часть данных) | Точная (полный доступ к базе) |
| **Поиск по смыслу** | Отличный (семантический поиск) | Хороший (полнотекстовый поиск) |
| **Агрегация данных** | Слабая (ограничен фрагментами) | Сильная (доступ ко всем записям) |
| **Настройка** | Простая (загрузил файлы — готово) | Требует определения инструментов |
| **Масштабируемость** | Хорошая (векторный поиск эффективен) | Ограничена размером памяти |
| **Гибкость запросов** | Ограничена семантическим поиском | Высокая (любые фильтры) |

**Рекомендации:**
- Используйте **RAG**, когда важен поиск по смыслу и работа с большими объёмами неструктурированных данных
- Используйте **Tool Calling**, когда нужны точные вычисления, агрегация и структурированный доступ к данным
- В реальных приложениях часто комбинируют оба подхода для получения наилучших результатов

## Освобождаем облачные ресурсы

Удалим из облака загруженные нами файлы и созданный индекс:

In [ ]:
print(f"🗑️  Удаляем индекс: {vector_store.id}")
client.vector_stores.delete(vector_store_id=vector_store.id)

print(f"🗑️  Удаляем {len(uploaded_chunks)} файлов")
for f in tqdm(uploaded_chunks):
    client.files.delete(file_id=f.id)

print("✅ Очистка завершена")